# 02 — Data collection guide (free sources)

This notebook is the practical companion to the README. It walks you through *how to actually populate* each of the three layers using free sources, with the search queries, URL patterns, and verification steps spelled out.

Use as a working notebook: edit it as you go, record what you find, commit incrementally. By the end of four focused weekends you should have a complete v1 dataset.

**Order matters.** Do them in this sequence:

1. **Weekend 1**: Verify Layer A (policy events) — fixed dates, primary sources
2. **Weekend 2**: Build Layer B corporate events — ASX direct for listed, trade press for unlisted
3. **Weekend 3**: AEMO Generator Information history — the project-level dataset
4. **Weekend 4**: Layer C information environment — trade press counts and conferences

Each weekend is sized for ~6 hours of focused work. Doing them in batches by event type / source is far faster than jumping around.

## Weekend 1: Verify Layer A policy events

Starting state: 42 events in `policy_events.csv`, 10 verified, 32 needing verification.

Goal: get all 42 to `confidence='verified'` with primary-source URLs and exact dates.

### Batching strategy

Do all events of one type before moving to the next. Reusing your context is much faster than constantly switching frames of reference.

**Batch 1 — NSW REZ designations (events e003-e006):**
- Primary source: NSW Government Gazette
- URL pattern: https://gazette.legislation.nsw.gov.au/ → search for "renewable energy zone" with date range
- Secondary: EnergyCo NSW per-REZ page (https://www.energyco.nsw.gov.au/renewable-energy-zones/...)
- Time estimate: 30 min for all 4 events

**Batch 2 — NSW REZ access scheme events (e008-e010, e041):**
- Primary source: EnergyCo NSW media releases page
- URL: https://www.energyco.nsw.gov.au/news
- Look for round openings, awards, and shortlist announcements
- Time estimate: 45 min

**Batch 3 — Federal CIS events (e011, e012, e039, e040):**
- Primary source: DCCEEW news page
- URL: https://www.dcceew.gov.au/about/news
- CIS auction announcements have specific dates published
- Time estimate: 30 min

**Batch 4 — AEMO publications (e013-e018):**
- Primary source: AEMO publication pages (each ISP/ESOO has its own landing page)
- URLs:
  - ISP: https://aemo.com.au/energy-systems/major-publications/integrated-system-plan-isp
  - ESOO: https://aemo.com.au/energy-systems/electricity/national-electricity-market-nem/nem-forecasting-and-planning/forecasting-and-reliability/nem-electricity-statement-of-opportunities-esoo
- Time estimate: 30 min

**Batch 5 — Transmission FIDs (e021-e026):**
- Primary sources: TNSP announcements + AER determinations
- TransGrid news: https://www.transgrid.com.au/about-us/news
- HumeLink: https://www.transgrid.com.au/projects-innovation/humelink
- Marinus Link: https://www.marinuslink.com.au/news
- EnergyConnect: https://www.transgrid.com.au/projects-innovation/project-energyconnect
- Time estimate: 60 min — more scattered sources

**Batch 6 — AEMC rule changes (e019, e020):**
- Primary source: AEMC rule change register
- URL: https://www.aemc.gov.au/regulation/rule-changes
- Filter by status and date — pull final determinations specifically
- Time estimate: 45 min

**Batch 7 — QLD events (e027-e033):**
- Primary source: DEPW QLD announcements + QLD Government Gazette
- QLD Energy and Jobs Plan: https://www.epw.qld.gov.au/about/initiatives/energy-and-jobs-plan
- LNP review materials: ministerial release pages from late 2024 onward
- Time estimate: 60 min

**Batch 8 — Annual MLF resets (e034-e037):**
- Primary source: AEMO MLF publication pages
- URL: https://aemo.com.au/energy-systems/electricity/national-electricity-market-nem/market-operations/marginal-loss-factors-mlf-information
- These are annual; published mid-year
- Time estimate: 20 min

Total: ~6 hours of focused work.

In [ ]:
# Run after editing the CSV to see what's still pending
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from nem_herding import load_events

events = load_events('../data/events/policy_events.csv')
pending = events[
    (events['confidence'] == 'needs_verify') | events['date_is_partial']
]
print(f'Still needing verification: {len(pending)} of {len(events)}')
print()
print('By event type:')
print(pending['event_type'].value_counts().to_string())

## Weekend 2: Layer B corporate events (ASX listed actors first)

Starting state: 20 corporate events seeded, all need verification.

Goal: 100-200 corporate events total by end of weekend, covering 2022-2025 for the major NSW and QLD VRE actors.

### Step 2.1 — ASX listed actors (highest signal, easiest to collect)

For each listed ticker in `actors.csv`, pull the last 3 years of market-sensitive announcements from ASX:

**URL pattern:** `https://www.asx.com.au/markets/company/{TICKER}/announcements`

Tickers to cover (from actors.csv):
- AGL.AX — AGL Energy
- ORG.AX — Origin Energy (pre-acquisition + post)
- GNX.AX — Genex Power (pre-takeover)

For each company:
1. Browse the announcements page
2. Filter to **market-sensitive only** (this filters out routine compliance disclosures)
3. Record any announcement matching: FID, commissioning, M&A, capital raise, strategic update, major contract
4. Skip: routine quarterly disclosures, change-of-director, AGM notices unless they contain substantive strategy content

**Time estimate:** 90 min for 3 listed actors covering 3 years.

### Step 2.2 — Major unlisted developers

For each unlisted actor in `actors.csv`, browse their media/news page:

Actors to cover this weekend (focus on the top 10 by NSW/QLD pipeline):
- Squadron Energy (https://www.squadronenergy.com/media)
- Acen Australia (https://www.acenrenewables.com.au/news)
- Lightsource bp (https://lightsourcebp.com/au/news/)
- Neoen Australia (https://neoen.com/en/news)
- Iberdrola Australia (https://www.iberdrola.com.au/press-room)
- Engie Australia (https://www.engie.com.au/about/news-and-media)
- Quinbrook (https://www.quinbrook.com/news)
- Octopus Investments (https://octopusinvestments.com.au/news)
- Ark Energy (https://www.arkenergy.com.au/news)
- Stanwell Corporation (https://www.stanwell.com/news)

Same filter: FIDs, commissionings, major project announcements, M&A, strategic updates. Skip the operational PR fluff.

**Time estimate:** 90 min for 10 unlisted actors.

### Step 2.3 — Trade press cross-reference

RenewEconomy is the highest-signal trade outlet. For each quarter from 2022-Q1 to 2025-Q2:

Search query pattern: `site:reneweconomy.com.au [REZ name] [year]`

Scan results for events that didn't show up in the company-direct searches (often unlisted developers announce via trade press first). Add to `corporate_events.csv`.

**Time estimate:** 90 min.

Total Weekend 2: ~4.5 hours. Should yield 100-200 events covering all major actors.

### Schema reminders when adding events

- `event_id`: continue the `cNNN` numbering (`c021`, `c022`, ...)
- `date`: `YYYY-MM-DD`, or `YYYY-MM-XX` if you only know the month
- `actor_id`: must exist in `actors.csv` (add the actor first if not)
- `event_type`: must be in the controlled vocabulary (see `corporate.py` for the full list)
- `source_type`: where you found it (`asx_disclosure`, `media_release`, `trade_press`)
- `confidence`: `verified` if from primary source with clear date; `needs_verify` if from secondary
- `rez`: comma-separated if multiple REZs affected
- `notes`: any context that helps later interpretation

In [ ]:
# Check progress on corporate events
from nem_herding import load_corporate_events

ce = load_corporate_events(
    '../data/events/corporate_events.csv',
    '../data/actors.csv',
)
print(f'Total corporate events: {len(ce)}')
print(f'Verified: {(ce["confidence"] == "verified").sum()}')
print(f'\nBy actor (top 10):')
print(ce["actor_name"].value_counts().head(10).to_string())
print(f'\nBy event type:')
print(ce["event_type"].value_counts().to_string())

## Weekend 3: AEMO Generator Information register

This is the project-level dataset that underpins the outcome variables (queue submission rate, commissioning rate, FID rate). It's the most analytically important single source.

### Step 3.1 — Download historical releases

AEMO publishes quarterly. URL: https://aemo.com.au/energy-systems/electricity/national-electricity-market-nem/participate-in-the-market/network-connections/generator-information-page

Download the Excel file for each quarter from 2020-Q1 to current. Save to `data/projects/` with naming `gen_info_YYYY_QN.xlsx`.

Older releases may not be on the current page — use the Wayback Machine:
- https://web.archive.org/web/2020*/https://aemo.com.au/energy-systems/electricity/national-electricity-market-nem/nem-forecasting-and-planning/forecasting-and-planning-data/generation-information

**Time estimate:** 60 min to track down and download ~20 quarterly releases.

### Step 3.2 — Parse and consolidate

Build `src/nem_herding/projects.py` (next module to write) that:

- Reads each quarterly Excel file
- Normalises the schema (AEMO changes column names occasionally)
- Produces a long-format DataFrame: one row per project per quarter, with status
- Identifies status transitions (e.g., 'committed' → 'under construction' → 'operating')
- Each transition is a dated event

**Time estimate:** 2-3 hours to write and debug.

### Step 3.3 — Tag projects to REZs

Each project has lat/lon (mostly) or at least a region/state. Tag to the appropriate REZ using:

1. **Exact match if lat/lon falls inside known REZ boundaries** — REZ shapefiles are publicly available from EnergyCo NSW and QLD planning bodies
2. **Project name heuristics** — many projects have REZ-revealing names (e.g., 'New England Solar Farm')
3. **Manual disambiguation** for the trickier ones

Save the tagged dataset to `data/projects/projects_tagged.parquet`.

**Time estimate:** 2-3 hours.

Total Weekend 3: 5-6 hours.

## Weekend 4: Layer C information environment

The hardest layer to collect. Approach in priority order — collect the most-leverage proxies first, then add others if time permits.

### Step 4.1 — Trade press article counts (highest priority)

For each outlet × REZ × month combination from 2020-01 to current:

1. Open browser, do Google site-search: `site:reneweconomy.com.au "[REZ name]" [year]`
2. Filter results by date range using Google's tools menu
3. Count articles per month
4. Record in `data/info_environment/trade_press_quarterly.csv`

**Important:** use the REZ's common-usage name in quotes, not the abbreviation. E.g., "Central-West Orana" not "CWO". Test the search before committing to the count - sometimes you need variants like "Central West Orana" (no hyphen).

**Recommended starter set:** RenewEconomy only, all 8 REZs, monthly counts for 2022-2025. That's 8 × 48 = 384 cells. About 4-5 hours of work at ~45 seconds per cell. Boring but it produces a unique dataset.

**v2 expansion (later):** add EcoGeneration, PV Magazine Australia, then AFR via state library access.

### Step 4.2 — Conference session counts (medium priority)

For each major conference × year:

1. Find the programme page (often archived if old)
2. Search for REZ-relevant sessions in titles and abstracts
3. Count per REZ
4. Record in `data/info_environment/conferences.csv`

Conferences to track:
- All-Energy Australia (annual, October)
- Australian Energy Week (annual, May/June)
- AFR Energy & Climate Summit (annual, October)
- Smart Energy Conference (annual, April)
- AUPEC (academic, November)
- EnergyNext (annual, July)

**Time estimate:** 1-2 hours.

### Step 4.3 — AEMO publication frequency (low priority for v1)

Count AEMO publications referencing each REZ per quarter. Their site has search but it's clunky. Defer to v2 unless time permits.

Total Weekend 4: 5-7 hours.

## After 4 weekends — what you'll have

A complete v1 data foundation:

- **Layer A**: ~50 verified policy events, primary-sourced
- **Layer B**: 150-250 corporate events tagged to actors and REZs
- **Project-level dataset**: every NEM-relevant generator with status history
- **Layer C**: monthly trade press intensity by REZ for 2022-2025, plus annual conference session counts

This is enough to:

1. Build the visual EDA (Notebook 04) — plot commissioning rate per REZ with policy events overlaid
2. Fit the first Hawkes regression — test the synchronisation threshold prediction (Notebook 05)
3. Test the Layer 1 vs Layer 2 coupling hypothesis — compare Hawkes response to direct vs informational events (Notebook 06)
4. Identify lead/lag relationships in actor activity (Notebook 07)

And it's all built from free, public, defensible sources.